# Adding Storages to the Energy System Model

In the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb), we initialized an energy system model, which defines the basic structure of the energy system such as locations, commodities, and the temporal resolution.

In this notebook, we introduce **Storages components**. Storages represent components that **can store a commodity and thus transfers it between time steps.**. We focus on the most essential parameters required to define and understand a Storage component, while more advanced and optional settings will be explained in subsequent notebooks.

Typical examples of storages include:

- Li-ion batteries to store electricity
- Depleted gas fields to store CO2
- Hot water tanks to store heat ...



## Initialize an energy system model

Before we can add sources, we need to initialize the energy system model as shown in the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb).

In [ ]:
import fine as fn  # Provides objects and functions to model an energy system
import pandas as pd  # Used to manage data in tables
import numpy as np  # Used to generate random input data
np.random.seed(42)  # Sets a "seed" to produce the same random input data in each model run

esM = fn.EnergySystemModel(
    locations = {"regionN", "regionS"},
    commodities = {"electricity", "naturalGas", "CO2"},
    commodityUnitsDict = {
    "electricity": r"GW$_{el}$",
    "naturalGas": r"GW$_{CH_{4},LHV}$",
    "CO2": r"Mio. t$_{CO_2}$/h",
    },
    costUnit = "1e6 Euro",
    lengthUnit = "km",
    numberOfTimeSteps = 8760,
    hoursPerTimeStep = 1
)

## Add Storages

### Lithium Ion Batteries

The self discharge of a lithium ion battery is here described as 3% per month. The self discharge per hours is obtained using the equation (1-$\text{selfDischarge}_\text{hour})^{30*24\text{h}} = 1-\text{selfDischarge}_\text{month}$.

We can now add Lithium Ion batteries as a storage Below you can find a more detailed explanation of the parameters used here.

In [ ]:
esM.add(
    fn.Storage(
        esM=esM,
        name="Li-ion batteries",
        commodity="electricity",
        hasCapacityVariable=True,
        chargeEfficiency=0.95,
        cyclicLifetime=10000,
        dischargeEfficiency=0.95,
        selfDischarge=1 - (1 - 0.03) ** (1 / (30 * 24)),
        chargeRate=1,
        dischargeRate=1,
        doPreciseTsaModeling=False,
        investPerCapacity=0.151,
        opexPerCapacity=0.002,
        interestRate=0.08,
        economicLifetime=22,
    )
)

## General Structure of a Storage Instance

The following code snippet shows the extensive structure of the `Storage` class and its arguments. Since the structure contains many parameters with varying levels of relevance, it is for clarity indicated next to each parameter whether it is defined in this notebook or not.

```python
Storage(
    esM,                    # defined in this notebook
    name,                   # defined in this notebook
    commodity,              # defined in this notebook
    chargeRate=1,           # defined in this notebook
    dischargeRate=1,        # defined in this notebook
    chargeEfficiency=1,     # defined in this notebook
    dischargeEfficiency=1,  # defined in this notebook
    selfDischarge=0,
    cyclicLifetime=None,
    stateOfChargeMin=0,
    stateOfChargeMax=1,
    hasCapacityVariable=True,
    capacityVariableDomain="continuous",
    capacityPerPlantUnit=1,
    hasIsBuiltBinaryVariable=False,
    bigM=None,
    doPreciseTsaModeling=False,
    chargeOpRateMax=None,   # defined in this notebook
    chargeOpRateFix=None,   # defined in this notebook
    chargeTsaWeight=1,
    dischargeOpRateMax=None,# defined in this notebook
    dischargeOpRateFix=None,# defined in this notebook
    dischargeTsaWeight=1,
    isPeriodicalStorage=False,
    locationalEligibility=None,
    capacityMin=None,
    capacityMax=None,
    partLoadMin=None,
    sharedPotentialID=None,
    linkedQuantityID=None,
    capacityFix=None,
    commissioningMin=None,
    commissioningMax=None,
    commissioningFix=None,
    isBuiltFix=None,
    investPerCapacity=0,
    investIfBuilt=0,
    opexPerChargeOperation=0,
    opexPerDischargeOperation=0,
    opexPerCapacity=0,
    opexIfBuilt=0,
    interestRate=0.08,
    economicLifetime=10,
    technicalLifetime=None,
    floorTechnicalLifetime=True,
    socOffsetDown=-1,
    socOffsetUp=-1,
    stockCommissioning=None,
    pwlcfParameters=None,
)
```
In the following sections, we explain the most important arguments of a Storage component.


## Required Arguments

### esM

`esM` is the energy system model to which the storage is added.

### name

`name` is a string, which should describe the type of storage which is added to the energy system model.

Examples:
- "Li-ion_battery"
- "salt_cavern"

### commodity

`commodity` defines which commodity the storage should store.

The commodity must be one of the commodities that were defined when initializing the `EnergySystemModel`.

Examples of commodities that could be stored include:<br>
- electricity from renewable generation
- hydrogen from external supply
- heat

### hasCapacityVariable

`hasCapacityVariable` is a **boolean**, which specifies whether the component has a capacity limit.

Examples:<br>
- A wind turbine has a capacity given in GW_electric -> ```hasCapacityVariable = True```
- Emitting CO2 into the environment is not per se limited by a capacity -> ```hasCapacityVariable = False```

## Optional Parameters : Technical Parameters

When it comes to storage, it is useful to further specify its technical characteristics in more detail. The following parameters define the main technical specifications related to storage components.

### charge/dischargeRate & charge/dischargeEfficiency

`chargeRate` and `dischargeRate` define the ratio of the maximum storage inflow and outflow (in commodityUnit/hour) to the storage capacity (in commodityUnit), respectively. 

`chargeEfficiency` and `dischargeEfficiency` define the efficiency with which the storage can be charged. This corresponds to the percentage of the injected commodity that is transformed into stored commodity. 

Example: 
A battery system has a storage capacity of 100 MWh_elec.
It can be charged at a maximum rate of 20 MWh_elec per hour, meaning the chargeRate = 20 / 100 = 0.2 h⁻¹.

The chargeEfficiency is 0.9, meaning that only 90% of the electricity taken is effectively stored, while 10% is lost during the charging process.


### Charge/DischargeOpRateMax & Charge/DischargeOpRateFix

`ChargeOpRateMax` and `DischargeOpRateMax` define a maximum charging rate for each location and each time step, if required also for each investment period, by a positive float. It depends on `hasCapacityVariable` as follows:

- If ```hasCapacityVariable = True```, the values are given relative to the installed capacities (i.e. a value of 1 indicates a utilization of 100% of the capacity).
- If ```hasCapacityVariable = False```, the values are given as absolute values in form of the `commodityUnit` for each time step.

Type:

- None (default)
- Pandas DataFrame with positive (>= 0) entries. The row indices have to match the in the energy system model specified time steps. The column indices have to equal the in the energy system model specified locations. The data in ineligible locations are set to zero.
- Dictionary with investment periods as keys and one of the two options above as values

### commodityCost

`commodityCost` describes the cost value of one operation's unit of the component. The cost unit in which the parameter is given has to match the one specified in the energy system model. 
The total cost is calculated as
$$
\text{commodityCost} \times \text{total annual operation.}
$$

Example: In a national energy system, natural gas could be purchased from another country with a certain cost.

Type: 
- positive float ($\geq 0$)
- Pandas Series with positive floats ($\geq 0$). The indices of the series have to equal the locations as specified in the energy system model.
- Dictionary with investment periods as keys and one of the two options above as values.

### capacityMax

`capacityMax` indicates the maximum capacity of this source.

Example: 

Type:
- None (default)
- float
- int
- Pandas Series with positive values ($\geq 0$). The indices of the series have to equal the in the energy system model specified locations (dimension=1dim) or connections between these locations in the format of 'loc1' + '_' + 'loc2' (dimension=2dim).
- Pandas DataFrame with positive values ($\geq 0$). The row and column indices of the DataFrame have to equal the in the energy system model specified locations.
- Dictionary with investment periods as keys and one of the options above as values.

### investPerCapacity

`investPerCapacity` describes the investment costs for one unit of the capacity. The invest of a component is obtained by multiplying the commissioned capacities of the component (in the physical Unit of the component) with the `investPerCapacity` factor and is distributed over the components technical lifetime. The value has to match the unit costUnit/physicalUnit (e.g. Euro/kW).

Example:

Type:
- float or Pandas Series with location specific values (dimension=1dim). The cost unit in which the parameter is given has to match the one specified in the energy system model (e.g. Euro, Dollar, 1e6 Euro). The value has to match the unit 
$$
\frac{\text{costUnit}}{\text{physicalUnit}}, \qquad \text{e.g.} \quad \frac{\text{Euro}}{\text{kW}}\text{, } \frac{\text{1e6 Euro}}{\text{GW}}.
$$
- float or Pandas Series or DataFrame with location specific values (dimension=2dim). The cost unit in which the parameter is given has to match the one specified in the energy system model divided by the specified lengthUnit (e.g. Euro/m, Dollar/m, 1e6 Euro/km). The value has to match the unit 
$$
\frac{\text{costUnit}}{\text{lengthUnit} \cdot \text{physicalUnit}}, \qquad \text{e.g.} \quad \frac{\text{Euro}}{\text{kW} \cdot \text{m}}\text{, } \frac{\text{1e6 Euro}}{\text{GW} \cdot \text{km}}.
$$
- Dictionary with years as keys (past years which had stock commissioning and investment periods which will be optimized) and one of the two options above as values, e.g. ```{2020: 1000, 2025: 800, 2030: 750}```

The default value is 0.

### opexPerCapacity

`opexPerCapacity` describes the operational cost for one unit of capacity. The annual operational cost, which are only a function of the capacity of the component (in the physicalUnit of the component) and not of the specific operation itself, are obtained by multiplying the commissioned capacity of the component at a location with the `opexPerCapacity` factor and is distributed over the components technical lifetime. The possible types are similar to the ones for [investPerCapacity](#investpercapacity).


### interestRate

`interestRate` describes the interest rate which is considered for computing the annuities of the invest of the component (depreciates the invests over the economic lifetime). A value of 0.08 corresponds to an interest rate of 8%. The interest rate is currently constant for all investment periods.
Warning: The interest must be greater than 0 if [annuityPerpetuity](../_01_initialize/_1_initialize_ESM.ipynb#annuityPerpetuity) is used in the energy system model.

Type:
- None
- Pandas Series with positive values ($\geq 0$). The indices of the series have to equal the in the energy system model specified locations (dimension=1dim) or connections between these locations in the format of 'loc1' + '_' + 'loc2' (dimension=2dim)
- Pandas DataFrame with positive values ($\geq 0$). The row and column indices of the DataFrame have to equal the in the energy system model specified locations.

### economicLifetime

`economicLifetime` describes the economic lifetime of the component which is considered for computing the annuities of the invest of the component (i.e. the depreciation time). The economic lifetime is currently constant over the pathway of investment periods.

Type:
- None
- Pandas Series with positive values ($\geq 0$). The indices of the series have to equal the in the energy system model specified locations (dimension=1dim) or connections between these locations in the format of 'loc1' + '_' + 'loc2' (dimension=2dim)
- Pandas DataFrame with positive ($\geq 0$) values. The row and column indices of the DataFrame have to equal the in the energy system model specified locations.

Many parameters were left out here. Some of them might need a page on their own. Others could be collected in an "other features" notebook